<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_40_middleware_redis/note_lesson_40_redis_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Урок 40 — Middleware і Redis: новинний агрегатор, крок 4

Агрегатор з уроку 39 зберігає новини в базі. Сьогодні додаємо те, що потрібно кожному API під навантаженням: **middleware** (код навколо кожного запиту), **кеш** стрічки в Redis, **rate limit** (`429`) і **фоновий збір** (`202`). Redis у ноутбуці — `fakeredis`, Redis у пам'яті процесу: сервер не потрібен.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | `news_hub`: база SQLite (Alembic) і Redis у пам'яті | 0.1 | |
| 1 | middleware: заголовки, журнал, порядок шарів, відповідь без ендпоінта | 1.1–1.5 | **Задача 1** |
| 2 | кеш стрічки: cache-aside, hit/miss, ключі, версія й інвалідація | 2.1–2.6 | **Задача 2** |
| 3 | rate limit: `429`, ключ без TTL, окремі лічильники, межа фіксованого вікна | 3.1–3.5 | |
| 4 | фоновий збір: `BackgroundTasks`, статус у Redis, `202`, `failed` | 4.1–4.4 | |
| 5 | ковзне вікно на sorted set | 5.1–5.2 | |
| 6 | знайди помилку: сесія запиту у фоні | 6.1 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, застосунок з маршрутами) лише створює об'єкти — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Задач **три**; перевірка кожної — у клітинці одразу під нею.
- Рефакторинги, діаграми і вивід на справжніх PostgreSQL і Redis — у книзі курсу: [Урок 40](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_40/).

---
## 0. Проєкт, база і Redis

`news_hub` з уроку 39 + три нові файли:

| Файл | Що в ньому |
|---|---|
| `cache.py` | клієнт Redis (`make_redis`, `get_redis`) і `NewsCache` — кеш стрічки з версією |
| `middleware.py` | `request_context` (id і час запиту), `rate_limit` (`429`), `invalidate_cache` (нова версія кешу після запису) |
| `jobs.py` | `JobStore` (статус задачі в Redis) і `run_scrape_job` — фоновий збір |

Змінні середовища — **до** імпорту `news_hub`: база — файл SQLite, Redis — `fakeredis://`.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_40_middleware_redis"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiosqlite, alembic, fakeredis, fastapi, redis, sqlalchemy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())

os.environ["DATABASE_URL"] = "sqlite+aiosqlite:///lesson40.db"
os.environ["REDIS_URL"] = "fakeredis://"
if os.path.exists("lesson40.db"):
    os.remove("lesson40.db")
subprocess.run([sys.executable, "-m", "alembic", "upgrade", "head"], capture_output=True, check=True)

import asyncio
import gc
import json
import logging
import time
import uuid
from functools import partial

import fakeredis
from fastapi import BackgroundTasks, FastAPI, HTTPException, Request, Response
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient

from news_hub.api import CacheDep, JobStore, NewsOut, RedisDep, RepoDep, app, cached_json, get_scrapers
from news_hub.cache import NewsCache
from news_hub.db import SessionFactory
from news_hub.middleware import RATE_LIMIT_REQUESTS, RATE_LIMIT_WINDOW, RateLimiter
from news_hub.models import validate_news
from news_hub.repository import NewsRepository
from news_hub.snapshot import load_snapshot

print("ліміт:", RATE_LIMIT_REQUESTS, "POST /api/scrape* за", RATE_LIMIT_WINDOW, "с")

### Приклад 0.1. Клієнт і Redis застосунку

`lifespan` створює клієнт Redis (`app.state.redis`). Він живе в циклі подій застосунку — у `TestClient` це окремий потік. Тому команди Redis з ноутбука надсилаємо через `client.portal.call(...)`: «виконай цю корутину там, де живе застосунок». Помічник `rcall` скорочує запис.

In [ ]:
client = TestClient(app)
_ = client.__enter__()
redis = app.state.redis                           # fakeredis, створений у lifespan
news_logger = logging.getLogger("news_hub")
news_logger.setLevel(logging.CRITICAL)            # журнал запитів покажемо в прикладі 1.2


def rcall(method, *args, **kwargs):
    """Виконати команду Redis застосунку: rcall(redis.get, "ключ")."""
    return client.portal.call(partial(method, *args, **kwargs))

In [ ]:
print(client.get("/health").json(), "| Redis:", type(redis).__name__, "| PING:", rcall(redis.ping))

`PING` → `True`: Redis відповідає. `FakeAsyncRedis` має ті самі методи, що `redis.asyncio.Redis`, — код агрегатора не знає, що Redis «несправжній».

---
## 1. Middleware

**Middleware** — функція, через яку проходить **кожен** запит до ендпоінта і **кожна** відповідь після нього:

```python
async def middleware(request: Request, call_next) -> Response:
    ...                                    # до ендпоінта
    response = await call_next(request)    # ендпоінт (і внутрішні шари)
    ...                                    # після: відповідь уже є
    return response
```

Туди виносять те, що стосується всіх ендпоінтів: журнал, час обробки, обмеження частоти, заголовки безпеки.

### Приклад 1.1. `X-Request-ID` і `X-Process-Time`

Middleware агрегатора `request_context` бере id запиту з заголовка клієнта (або генерує свій) і додає у відповідь id і час обробки.

In [ ]:
for headers in ({"X-Request-ID": "my-first-request"}, {}):
    response = client.get("/health", headers=headers)
    print(response.status_code, response.headers["X-Request-ID"], response.headers["X-Process-Time"])
assert client.get("/health", headers={"X-Request-ID": "abc"}).headers["X-Request-ID"] == "abc"

Свій id повернувся без змін; без заголовка — згенерований (8 символів uuid). За цим id у журналі знаходять усе, що стосується одного запиту (урок 50). Ендпоінт `/health` про id нічого не знає.

### Приклад 1.2. Журнал запитів

`request_context` пише рядок у журнал `news_hub` для кожного запиту (`lifespan` підключив до нього вивід у stderr). На початку ноутбука журнал приглушено; увімкнемо його на три запити.

In [ ]:
news_logger.setLevel(logging.INFO)
client.get("/health", headers={"X-Request-ID": "log-demo"})
client.get("/api/news/count")
client.get("/no-such-page")
news_logger.setLevel(logging.CRITICAL)     # далі журнал вимкнено, щоб не заважав виводу

Рядок на кожен запит: id, метод, шлях, статус, час — навіть для `404`, до якого не дійшов жоден ендпоінт. Middleware бачить **усі** запити, тому журнал і метрики пишуть саме тут.

### Приклад 1.3. Свій middleware: заголовок у кожній відповіді

Найпростіший middleware: нічого не змінює в запиті, лише додає заголовок у відповідь. `app.middleware("http")(функція)` — реєстрація (те саме, що декоратор `@app.middleware("http")`).

In [ ]:
headers_app = FastAPI()


async def add_lesson_header(request: Request, call_next) -> Response:
    response = await call_next(request)
    response.headers["X-Lesson"] = "40"
    return response


headers_app.middleware("http")(add_lesson_header)


@headers_app.get("/a")
async def page_a() -> dict[str, str]:
    return {"page": "a"}


@headers_app.get("/b")
async def page_b() -> dict[str, str]:
    return {"page": "b"}

In [ ]:
headers_client = TestClient(headers_app)
for path in ("/a", "/b", "/missing"):
    response = headers_client.get(path)
    print(path, response.status_code, response.headers.get("X-Lesson"))

Заголовок отримали **всі** відповіді, включно з `404`, хоча жоден ендпоінт його не ставив. Middleware реєструють **до** першого запиту: після старту застосунку Starlette не дозволить додати новий.

**🔮 Прогноз:** Два middleware: `outer` зареєстровано **останнім**, `inner` — першим. Кожен дописує в список `trace` рядок до `call_next` і після. Який вийде список?

<details>
<summary>Відповідь</summary>

`['outer до', 'inner до', 'ендпоінт', 'inner після', 'outer після']` — останній зареєстрований стає зовнішнім шаром «цибулі».

</details>

### Приклад 1.4. Порядок шарів: «цибуля»

In [ ]:
onion = FastAPI()
trace: list[str] = []


async def inner(request: Request, call_next) -> Response:
    trace.append("inner до")
    response = await call_next(request)
    trace.append("inner після")
    return response


async def outer(request: Request, call_next) -> Response:
    trace.append("outer до")
    response = await call_next(request)
    trace.append("outer після")
    return response


onion.middleware("http")(inner)        # першим
onion.middleware("http")(outer)        # останнім → зовнішній шар


@onion.get("/")
async def onion_root() -> dict[str, str]:
    trace.append("ендпоінт")
    return {"ok": "yes"}

In [ ]:
trace.clear()
TestClient(onion).get("/")
print(trace)
assert trace == ["outer до", "inner до", "ендпоінт", "inner після", "outer після"]

Запит іде всередину від останнього зареєстрованого, відповідь — назовні у зворотному порядку. Тому в `api.py` реєстрація «знизу вгору»:

```python
app.middleware("http")(invalidate_cache)   # внутрішній
app.middleware("http")(rate_limit)
app.middleware("http")(request_context)    # зовнішній: його час включає всі інші шари
```

### Приклад 1.5. Відповідь без ендпоінта

Middleware може **не викликати** `call_next` — тоді ендпоінт не виконається. Так працює rate limit (розділ 3), а тут — «режим обслуговування»: поки прапорець увімкнено, усі запити, крім `/health`, отримують `503`.

In [ ]:
maintenance_app = FastAPI()
MAINTENANCE = {"on": False}
calls: list[str] = []


async def maintenance(request: Request, call_next) -> Response:
    if MAINTENANCE["on"] and request.url.path != "/health":
        return JSONResponse(status_code=503, content={"detail": "оновлюємо агрегатор, спробуй за хвилину"},
                            headers={"Retry-After": "60"})
    return await call_next(request)


maintenance_app.middleware("http")(maintenance)


@maintenance_app.get("/news")
async def maintenance_news() -> list[str]:
    calls.append("ендпоінт /news")
    return ["Новина 1", "Новина 2"]


@maintenance_app.get("/health")
async def maintenance_health() -> dict[str, str]:
    return {"status": "ok"}

In [ ]:
maintenance_client = TestClient(maintenance_app)
calls.clear()
for on in (False, True):
    MAINTENANCE["on"] = on
    news, health = maintenance_client.get("/news"), maintenance_client.get("/health")
    print(f"обслуговування={on!s:5} /news → {news.status_code} {news.json()} | /health → {health.status_code}")
print("ендпоінт викликали:", calls)

У режимі обслуговування `/news` отримав `503` і `Retry-After`, а ендпоінт виконався лише **один** раз — у першому запиті. `/health` пропускаємо завжди: за ним балансувальник і Docker перевіряють, чи живий сервер (уроки 49–50).

### Задача 1. Ключ для `/admin`

Напиши middleware `require_api_key`:

- шлях починається з `/admin` і заголовок `X-API-Key` **не дорівнює** `API_KEY` → відповідь `JSONResponse` зі статусом `401` і тілом `{"detail": "потрібен ключ X-API-Key"}`, **без** виклику ендпоінта;
- інакше — звичайний `await call_next(request)`.

Застосунок для перевірки будує функція `make_guarded_app(middleware)` — вона вже є в клітинці нижче. Перевірка — у клітинці після задачі.

In [ ]:
API_KEY = "lesson-40-key"


def make_guarded_app(middleware) -> FastAPI:
    """Застосунок з /public і /admin/stats, обгорнутий переданим middleware."""
    guarded = FastAPI()
    guarded.middleware("http")(middleware)

    @guarded.get("/public")
    async def public() -> dict[str, str]:
        return {"page": "public"}

    @guarded.get("/admin/stats")
    async def admin_stats() -> dict[str, int]:
        return {"news": 168}

    return guarded

In [ ]:
async def require_api_key(request: Request, call_next) -> Response:
    # YOUR CODE HERE
    pass


In [ ]:
guarded_client = TestClient(make_guarded_app(require_api_key), raise_server_exceptions=False)
public = guarded_client.get("/public")
assert public.status_code == 200, f"/public → {public.status_code}: middleware має повертати відповідь call_next"
denied = guarded_client.get("/admin/stats")
assert denied.status_code == 401, f"/admin/stats без ключа → {denied.status_code}, а треба 401"
assert denied.json() == {"detail": "потрібен ключ X-API-Key"}, denied.json()
assert guarded_client.get("/admin/stats", headers={"X-API-Key": "wrong"}).status_code == 401
allowed = guarded_client.get("/admin/stats", headers={"X-API-Key": API_KEY})
assert allowed.status_code == 200 and allowed.json() == {"news": 168}, allowed.text
print(public.json(), "|", denied.status_code, denied.json(), "|", allowed.json())
print("✅ Задача 1 розв'язана")

---
## 2. Кеш стрічки

`GET /api/news` і `/api/news/stats` читають однакові дані сотні разів, а змінюються вони лише під час збору. Це **cache-aside** з уроку 31: є в кеші (hit) — віддати; немає (miss) — порахувати, покласти в кеш з TTL, віддати.

### Приклад 2.1. Cache-aside вручну

Спершу — без FastAPI, на окремому Redis у пам'яті. `slow_stats()` «рахує» 0,3 с; `cached_stats()` спершу дивиться в кеш.

In [ ]:
demo_redis = fakeredis.FakeAsyncRedis(decode_responses=True)


async def slow_stats() -> dict[str, int]:
    await asyncio.sleep(0.3)                       # «важкий» запит до бази
    return {"uk": 30, "ru": 138}


async def cached_stats() -> tuple[dict[str, int], str]:
    if (cached := await demo_redis.get("demo:stats")) is not None:
        return json.loads(cached), "HIT"
    value = await slow_stats()
    await demo_redis.set("demo:stats", json.dumps(value), ex=60)   # ex — TTL у секундах
    return value, "MISS"

In [ ]:
for attempt in (1, 2, 3):
    start = time.perf_counter()
    value, state = await cached_stats()
    print(attempt, state, value, f"{time.perf_counter() - start:.2f} с")
print("TTL ключа:", await demo_redis.ttl("demo:stats"), "с")

Перший раз — `MISS` і 0,3 с; далі — `HIT` за соті частки секунди. У кеші лежить **готовий JSON** (рядок), а TTL — 60 с: найдовше, скільки дані можуть «відставати» від бази.

### Приклад 2.2. `X-Cache` у агрегаторі

Збираємо знімок, потім тричі однаковий запит і раз — з іншими параметрами. Заголовок `X-Cache` каже, звідки відповідь.

In [ ]:
client.post("/api/scrape", json={"source": "snapshot"})
states = []
for attempt in (1, 2, 3):
    response = client.get("/api/news", params={"lang": "uk", "limit": 3})
    states.append(response.headers["X-Cache"])
    print(attempt, response.headers["X-Cache"], len(response.json()), "новини")
print("інші параметри:", client.get("/api/news", params={"lang": "ru", "limit": 3}).headers["X-Cache"])
assert states == ["MISS", "HIT", "HIT"]

`?lang=uk&limit=3` і `?lang=ru&limit=3` — **різні** записи кешу: ключ залежить від параметрів. Під час `HIT` ендпоінт не звертався ні до бази, ні до Pydantic — віддав рядок з Redis.

### Приклад 2.3. Ключ не залежить від порядку параметрів

`NewsCache.key(name, params)` будує ключ `news:v<версія>:<name>:<хеш параметрів>`. Параметри перетворюються на JSON з **відсортованими** ключами, тож `?lang=uk&limit=3` і `?limit=3&lang=uk` — один ключ.

In [ ]:
cache = NewsCache(redis)
key_a = rcall(cache.key, "list", {"lang": "uk", "limit": 3})
key_b = rcall(cache.key, "list", {"limit": 3, "lang": "uk"})
key_c = rcall(cache.key, "list", {"lang": "ru", "limit": 3})
print(key_a, key_b, key_c, sep="\n")
assert key_a == key_b != key_c and key_a.startswith("news:v")

Перші два ключі однакові, третій — інший хеш. Короткий SHA-1 (12 символів) замість самих параметрів — ключ завжди однієї довжини і без «дивних» символів з рядка запиту.

### Приклад 2.4. Що лежить у Redis

In [ ]:
print("news:version =", rcall(redis.get, "news:version"))
for key in sorted(rcall(redis.keys, "news:v[0-9]*")):
    print(key, "TTL", rcall(redis.ttl, key), "с")

Два записи стрічки (`uk` і `ru`) з TTL ≈ 60 с, і поточна версія. Версія — `1`, бо `POST /api/scrape` вище вже раз її збільшив (middleware `invalidate_cache`).

`KEYS` — лише для навчання: на великій базі ця команда блокує Redis, поки перебирає **всі** ключі.

**🔮 Прогноз:** Після `POST /api/news` (201) що покаже `X-Cache` для `/api/news/stats`, закешованого перед тим? А після невдалого `POST` (409)?

<details>
<summary>Відповідь</summary>

Після 201 — `MISS`: middleware `invalidate_cache` збільшив `news:version`, старий ключ більше не читається. Після 409 — `HIT`: запис не відбувся, версія не змінилась.

</details>

### Приклад 2.5. Інвалідація після запису

In [ ]:
NEW = {"title": "НБУ залишив облікову ставку без змін", "url": "https://www.rbc.ua/ukr/news/nbu-rate-778.html"}

In [ ]:
print("до:   ", client.get("/api/news/stats").headers["X-Cache"], client.get("/api/news/stats").headers["X-Cache"])
created = client.post("/api/news", json=NEW)
stats = client.get("/api/news/stats")
print("після:", created.status_code, stats.headers["X-Cache"], "total =", stats.json()["total"])
conflict = client.post("/api/news", json=NEW)
print("409:  ", conflict.status_code, client.get("/api/news/stats").headers["X-Cache"])
assert (stats.headers["X-Cache"], conflict.status_code) == ("MISS", 409)

Після `201` — `MISS` і `total = 169`: кеш не віддав застарілу статистику. Після `409` — `HIT`: дані не змінились, кеш лишився.

Чому версію збільшує **middleware**, а не ендпоінт: `COMMIT` робить `get_db` після ендпоінта. Якби версія росла в ендпоінті, паралельний `GET` між «версія +1» і `COMMIT` прочитав би ще **старі** дані й поклав їх у кеш під **новою** версією — на 60 с. Middleware отримує відповідь, коли `COMMIT` уже відбувся.

### Приклад 2.6. Які запити змінюють версію

Правило `invalidate_cache`: **успішний** (`< 400`) `POST`/`PATCH`/`PUT`/`DELETE` до `/api/news…` або `/api/scrape` → `INCR news:version`. Перевіримо кілька запитів.

In [ ]:
def version() -> int:
    return int(rcall(redis.get, "news:version") or 0)


news_id = created.json()["id"]
for label, send in [
    ("GET стрічки", lambda: client.get("/api/news")),
    ("PATCH 200", lambda: client.patch(f"/api/news/{news_id}", json={"category": "Економіка"})),
    ("PATCH 422", lambda: client.patch(f"/api/news/{news_id}", json={"category": "x"})),
    ("PATCH 404", lambda: client.patch("/api/news/9999", json={"category": "Спорт"})),
    ("DELETE 204", lambda: client.delete(f"/api/news/{news_id}")),
]:
    before = version()
    status = send().status_code
    print(f"{label:12} → {status} | версія {before} → {version()}")

Версія зростає лише після успішних записів — `PATCH 200` і `DELETE 204`. `GET` нічого не змінює; `422` і `404` — запис не відбувся, кеш правильний. Усі старі ключі після `INCR` просто перестають читатися, а TTL їх прибере — не треба шукати й видаляти ключі за шаблоном.

### Задача 2. Кеш картки новини

Додай ендпоінт `GET /api/news/{news_id}/card` — картка новини з кешем:

1. **Ключ** — `await cache.key("item", {"id": news_id})`: версія спільна, тож будь-який запис у новини скидає й картки.
2. **Hit** — `return cached_json(cached, "HIT")`.
3. **Miss** — `row = await repo.get(news_id)`; немає — `HTTPException(404, detail=f"новини {news_id} немає")`, і **`404` не кешуємо**: інакше новина, створена за секунду, 60 с виглядатиме відсутньою.
4. **Запис** — `payload = NewsOut.model_validate(row).model_dump_json()`, `await cache.set(key, payload)`, `return cached_json(payload, "MISS")`.

Аргументи ендпоінта: `news_id: int, repo: RepoDep, cache: CacheDep`. Перевірка — у наступній клітинці.

In [ ]:
# YOUR CODE HERE


In [ ]:
first = client.get("/api/news/1/card")
assert first.status_code == 200, f"GET /api/news/1/card → {first.status_code}: ендпоінт ще не додано?"
second = client.get("/api/news/1/card")
assert (first.headers.get("X-Cache"), second.headers.get("X-Cache")) == ("MISS", "HIT"), "перший запит — MISS, другий — HIT"
assert second.json()["id"] == 1 and second.json() == first.json()
keys_before = set(rcall(redis.keys, "news:v*"))
assert client.get("/api/news/9999/card").status_code == 404
assert set(rcall(redis.keys, "news:v*")) == keys_before, "відповідь 404 потрапила в кеш — її кешувати не можна"
client.patch("/api/news/1", json={"category": "Політика"})
third = client.get("/api/news/1/card")
assert third.headers["X-Cache"] == "MISS" and third.json()["category"] == "Політика"
print(first.headers["X-Cache"], second.headers["X-Cache"], "| після PATCH:", third.headers["X-Cache"], third.json()["category"])
print("✅ Задача 2 розв'язана")

---
## 3. Rate limit

Не більше `RATE_LIMIT_REQUESTS` (5) запитів `POST /api/scrape*` за `RATE_LIMIT_WINDOW` (60) секунд з однієї адреси. Лічильник — ключ Redis `rate:scrape:<адреса>` з TTL; адреса клієнта `TestClient` — `testclient`.

`RateLimiter.hit` — **одна транзакція** (`MULTI … EXEC`):

```python
async with self._redis.pipeline(transaction=True) as pipe:
    pipe.incr(key)
    pipe.expire(key, self.window, nx=True)      # TTL лише новому ключу
    pipe.ttl(key)
    count, _, ttl = await pipe.execute()
return count <= self.limit, count, max(ttl, 0)
```

### Приклад 3.1. До `429`

Скинемо лічильник (у розділі 2 вже був один збір) і надсилатимемо збори, поки не отримаємо `429`.

In [ ]:
def reset_rate_limit() -> None:
    """Скинути лічильник зборів для клієнта TestClient."""
    rcall(redis.delete, "rate:scrape:testclient")

In [ ]:
reset_rate_limit()
statuses = []
while True:
    response = client.post("/api/scrape", json={"source": "snapshot"})
    statuses.append(response.status_code)
    print(len(statuses), response.status_code, "залишилось:", response.headers.get("X-RateLimit-Remaining"),
          "| Retry-After:", response.headers.get("Retry-After"))
    if response.status_code == 429:
        retry_after = int(response.headers["Retry-After"])
        break
print(response.json()["detail"])
print("GET не обмежується:", client.get("/api/news/count").status_code)
assert statuses == [200] * RATE_LIMIT_REQUESTS + [429]
assert 0 < retry_after <= RATE_LIMIT_WINDOW

П'ять `200`, шостий — `429 Too Many Requests` з `Retry-After` (скільки секунд чекати) і поясненням у `detail`. Запит з `429` до ендпоінта не дійшов: `rate_limit` відповів сам, не викликаючи `call_next`. `GET` не обмежується — правило лише для дорогих зборів.

### Приклад 3.2. Ключ без TTL — блокування назавжди

Код-попередник (`ai_bot`) ставив TTL **окремою** командою і лише при `count == 1`. Якщо процес падав між `INCR` і `EXPIRE`, ключ лишався без TTL: наступні `INCR` дають 2, 3, 4… — і `count == 1` більше не спрацює ніколи. «Падіння» імітуємо, просто не викликаючи `EXPIRE`.

In [ ]:
async def separate_commands(key: str) -> tuple[str | None, int]:
    await demo_redis.delete(key)
    await demo_redis.incr(key)                   # INCR… і процес «упав» до EXPIRE
    for _ in range(3):
        count = await demo_redis.incr(key)
        if count == 1:                           # ніколи не виконається
            await demo_redis.expire(key, 60)
    return await demo_redis.get(key), await demo_redis.ttl(key)

In [ ]:
print("окремі команди: count, TTL =", await separate_commands("rate:old"))

await demo_redis.set("rate:scrape:crashed", 99)          # той самий «залишок» без TTL
print("TTL залишку:", await demo_redis.ttl("rate:scrape:crashed"))
allowed, count, ttl_after = await RateLimiter(demo_redis).hit("crashed", "scrape")
print("транзакція: count =", count, "| дозволено:", allowed, "| TTL =", ttl_after)
assert ttl_after > 0 and not allowed

`TTL = -1` — ключ без часу життя: клієнта заблоковано назавжди. `EXPIRE … NX` ставить TTL, **якщо його немає**, і робить це в тій самій транзакції, що `INCR`, — навіть «залишок» після збою сам зникне за хвилину.

### Приклад 3.3. У кожного клієнта свій лічильник

Ключ — `rate:<дія>:<клієнт>`. Ліміт 2 на клієнта: двоє клієнтів по три запити.

In [ ]:
limiter = RateLimiter(demo_redis, limit=2, window=60)
for client_ip in ("10.0.0.1", "10.0.0.2"):
    await demo_redis.delete(f"rate:scrape:{client_ip}")

In [ ]:
for client_ip in ("10.0.0.1", "10.0.0.2"):
    results = [await limiter.hit(client_ip, "scrape") for _ in range(3)]
    print(client_ip, [(allowed, count) for allowed, count, _ in results])
print("інша дія:", await limiter.hit("10.0.0.1", "create"))

Кожен клієнт отримав свої 2 дозволи, третій запит — ні. Інша дія (`create`) рахується окремо: обмеження зборів не заважає створювати новини. Відхилені запити теж збільшують лічильник (`count = 3`) — інакше «настирливий» клієнт нічого б не втрачав.

### Приклад 3.4. Межа фіксованого вікна

`RateLimiter` — **фіксоване вікно**: лічильник живе, поки живе ключ, і обнуляється разом із ним. На межі двох вікон можна пройти майже вдвічі більше запитів. Ліміт 3 за 2 с: перший запит відкрив вікно, чекаємо його кінця і пів секунди шлемо запит кожні 50 мс.

In [ ]:
async def window_edge() -> int:
    await demo_redis.delete("rate:edge:demo")
    edge_limiter = RateLimiter(demo_redis, limit=3, window=2)
    await edge_limiter.hit("demo", "edge")                              # 1-й запит відкрив вікно
    while await demo_redis.pttl("rate:edge:demo") > 150:                # чекаємо кінця вікна
        await asyncio.sleep(0.05)
    start, allowed = time.perf_counter(), 0
    while time.perf_counter() - start < 0.5:                            # пів секунди — запит кожні 50 мс
        allowed += (await edge_limiter.hit("demo", "edge"))[0]
        await asyncio.sleep(0.05)
    return allowed

In [ ]:
passed = await window_edge()
print(f"за 0.5 с пройшло {passed} запитів при ліміті 3 за 2 с")
assert passed > 3

Більше трьох за пів секунди: частина пройшла в кінці старого вікна, частина — на початку нового. Для захисту від спаму зборами цього досить — простота важить більше. Де потрібна точність (платне API), беруть **ковзне вікно** (розділ 5).

### Приклад 3.5. Скільки лишилося: `X-RateLimit-Remaining`

Клієнт не мусить «вгадувати» ліміт: кожна відповідь на збір несе `X-RateLimit-Limit` і `X-RateLimit-Remaining`. Перевіримо після скидання лічильника.

In [ ]:
reset_rate_limit()
response = client.post("/api/scrape", json={"source": "snapshot"})
print({name: response.headers[name] for name in ("X-RateLimit-Limit", "X-RateLimit-Remaining")})
print("лічильник у Redis:", rcall(redis.get, "rate:scrape:testclient"), "| TTL:", rcall(redis.ttl, "rate:scrape:testclient"))

Ліміт 5, лишилось 4; у Redis — лічильник `1` з TTL ≈ 60 с. Добрий клієнт (бот, фронтенд) сповільнюється сам, щойно `Remaining` наближається до нуля.

---
## 4. Фоновий збір

Живий збір — сім сторінок rbc.ua з тайм-аутом 15 с кожна. Тримати HTTP-запит відкритим стільки часу погано: клієнт може відвалитися за тайм-аутом. Рішення: відповісти **одразу** `202 Accepted` («прийнято до роботи») і працювати у фоні.

### Приклад 4.1. `BackgroundTasks`: спершу відповідь, потім задача

`background.add_task(функція, аргументи)` — FastAPI виконає функцію **після** того, як відправить відповідь.

In [ ]:
bg_app = FastAPI()
bg_events: list[str] = []


async def write_report(name: str) -> None:
    await asyncio.sleep(0.1)
    bg_events.append(f"фон: звіт «{name}» готовий")


@bg_app.post("/reports", status_code=202)
async def start_report(background: BackgroundTasks) -> dict[str, str]:
    background.add_task(write_report, "ранковий")
    bg_events.append("ендпоінт: відповідь 202")
    return {"status": "accepted"}

In [ ]:
bg_events.clear()
response = TestClient(bg_app).post("/reports")
print(response.status_code, response.json())
print(bg_events)

Ендпоінт завершився **до** задачі. На справжньому сервері клієнт отримує `202` одразу, а задача працює далі. `TestClient` для зручності тестів дочікується фонових задач, перш ніж повернути відповідь, — тому тут статус задачі вже фінальний; на сервері його треба **опитувати** (книга).

### Приклад 4.2. Статус задачі в Redis: `JobStore`

Статус — hash `job:<id>` з TTL на добу: `status`, `news_found`, `news_saved`, `error`, час. Скористаємось `JobStore` напряму.

In [ ]:
async def job_demo() -> None:
    jobs = JobStore(demo_redis)
    job = await jobs.create("snapshot", "async")
    print("створено:", job.status, job.job_id)
    await jobs.update(job.job_id, status="running")
    await jobs.update(job.job_id, status="done", news_found=168, news_saved=168)
    print("hash у Redis:", await demo_redis.hgetall(f"job:{job.job_id}"))
    print("TTL:", await demo_redis.ttl(f"job:{job.job_id}"), "с")
    print("модель:", (await jobs.get(job.job_id)).status, "| немає такої:", await jobs.get("nope"))

In [ ]:
await job_demo()

У Redis усе — рядки (`'168'`), а `JobStore.get` перетворює hash назад на модель `JobStatus` з числами. TTL — 86 400 с (доба): старі статуси прибираються самі. Неіснуюча задача — `None`, ендпоінт перетворює це на `404`.

### Приклад 4.3. `POST /api/scrape/jobs` → `202`

Фоновий збір теж `POST /api/scrape…` — тож підлягає rate limit; скинемо лічильник. Очистимо новини, запустимо збір знімка у фоні й прочитаємо статус.

In [ ]:
reset_rate_limit()
client.delete("/api/news")
started = client.post("/api/scrape/jobs", json={"source": "snapshot"})
job = client.get(f"/api/scrape/jobs/{started.json()['job_id']}").json()
print(started.status_code, {key: started.json()[key] for key in ("status", "news_saved")})
print({key: job[key] for key in ("status", "news_found", "news_saved", "error")})
print("у базі:", client.get("/api/news/count").json())
assert started.status_code == 202 and job["status"] == "done" and job["news_saved"] == 168

Відповідь `202` прийшла зі статусом `queued`; коли ми запитали статус — уже `done` і 168 новин у базі. `run_scrape_job` відкрив **свою** сесію бази, зробив `COMMIT` і лише потім збільшив версію кешу.

### Приклад 4.4. Помилка у фоні не губиться

Фонову задачу ніхто не чекає: якщо вона впаде, виняток ніхто не побачить. Тому `run_scrape_job` ловить помилку і записує її в статус `failed`. Підмінимо скрапер на «зламаний» (як rbc.ua, що не відповідає) і запустимо живий збір.

In [ ]:
async def broken_scraper(pages: list[str] | None):
    raise ConnectionError("rbc.ua не відповідає")

In [ ]:
app.dependency_overrides[get_scrapers] = lambda: {"async": broken_scraper, "sequential": broken_scraper}
try:
    started = client.post("/api/scrape/jobs", json={"source": "live"})
    failed = client.get(f"/api/scrape/jobs/{started.json()['job_id']}").json()
finally:
    app.dependency_overrides.clear()
print(started.status_code, {key: failed[key] for key in ("status", "news_found", "error")})
assert failed["status"] == "failed" and "ConnectionError" in failed["error"]

Клієнт отримав `202` (задачу прийнято), а статус чесно каже `failed` і причину. Без `try/except` у задачі статус назавжди лишився б `running`.

`BackgroundTasks` виконуються в тому самому процесі: якщо сервер перезапуститься, задача пропаде. Для надійних черг — окремий worker (Celery, RQ, arq; уроки 49–50).

---
## 5. Ковзне вікно

Фіксоване вікно на межі пропускає до 2× ліміту (приклад 3.4). **Ковзне** рахує запити за останні `window` мілісекунд на **sorted set** (урок 31): елемент — унікальний id запиту, бал — час запиту. В одній транзакції:

1. `ZREMRANGEBYSCORE` — прибрати запити, старші за `now - window`;
2. `ZADD` — додати поточний;
3. `ZCARD` — скільки у вікні;
4. `PEXPIRE` — ключ зникне, коли все застаріє.

Дозволено, якщо `ZCARD <= limit`. Час передаємо аргументом `now_ms` — так приклад не залежить від швидкості машини.

### Приклад 5.1. `sliding_hit`

In [ ]:
sliding_redis = fakeredis.FakeAsyncRedis(decode_responses=True)


async def sliding_hit(key: str, limit: int, window_ms: int, now_ms: int) -> bool:
    async with sliding_redis.pipeline(transaction=True) as pipe:
        pipe.zremrangebyscore(key, 0, now_ms - window_ms)
        pipe.zadd(key, {uuid.uuid4().hex: now_ms})
        pipe.zcard(key)
        pipe.pexpire(key, window_ms)
        _, _, count, _ = await pipe.execute()
    return count <= limit

In [ ]:
# ліміт 3 за 2000 мс: запит на 0 мс, потім серія на «межі» 1900–2150 мс
times = [0, 1900, 1950, 2050, 2100, 2150]
results = [await sliding_hit("rate:sliding:demo", 3, 2000, t) for t in times]
print(list(zip(times, results)))
assert results == [True, True, True, True, False, False]

- 0, 1900, 1950 мс — у вікні 1, 2, 3 запити → дозволено;
- 2050 мс: запит з 0 мс уже поза вікном `(50, 2050]` → у вікні 1900, 1950, 2050 = 3 → дозволено;
- 2100 і 2150 мс: у вікні 4 і 5 (відхилені теж рахуються) → ні.

У будь-які 2 с — не більше 3 дозволених. Фіксоване вікно на тій самій серії пропустило б більше.

### Приклад 5.2. Що лежить у sorted set

In [ ]:
for member, score in await sliding_redis.zrange("rate:sliding:demo", 0, -1, withscores=True):
    print(f"{member[:8]}…  час {int(score)} мс")
print("PTTL:", await sliding_redis.pttl("rate:sliding:demo"), "мс")

У множині — п'ять запитів з балами-часом (0 мс уже прибрано). Ціна точності — **пам'ять**: фіксоване вікно тримає одне число на клієнта, ковзне — по елементу на кожен запит у вікні. Для ліміту «5 за хвилину» це дрібниця, для «10 000 за годину» — вже ні.

---
## 6. Знайди помилку: сесія запиту у фоні

Колега вирішив, що окрема сесія для фону — зайве ускладнення, і передав у фонову задачу **репозиторій запиту** (як прототип `news_dashboard` передавав у задачу `db` запиту).

### Приклад 6.1. «Збережено 168» — а в базі порожньо

In [ ]:
async def scrape_in_background(repo: NewsRepository, jobs: JobStore, job_id: str) -> None:
    valid, _ = validate_news(load_snapshot())
    saved = await repo.add_many(valid)
    await jobs.update(job_id, status="done", news_saved=saved)


@app.post("/api/scrape/jobs-buggy", status_code=202)
async def start_buggy(background: BackgroundTasks, repo: RepoDep, redis: RedisDep) -> dict[str, str]:
    job = await JobStore(redis).create("snapshot", "async")
    background.add_task(scrape_in_background, repo, JobStore(redis), job.job_id)
    return {"job_id": job.job_id}

In [ ]:
reset_rate_limit()
client.delete("/api/news")
job_id = client.post("/api/scrape/jobs-buggy").json()["job_id"]
print("задача:", client.get(f"/api/scrape/jobs/{job_id}").json()["news_saved"], "збережено")
print("у базі:", client.get("/api/news/count").json())

# «Воскресла» сесія досі тримає незакомічену транзакцію — у SQLite це блокування всієї бази на запис
# («database is locked» для наступного INSERT). Приберемо її збирачем сміття, щоб продовжити.
_ = gc.collect()

Задача звітує «збережено 168», а в базі — 0. Попередження SQLAlchemy про «non-checked-in connection» під виводом клітинки — це і є забуте з'єднання тієї сесії: його прибирає збирач сміття, і незакомічена транзакція відкочується. Фонова задача виконується **після** відповіді: `get_db` уже зробив `COMMIT` своєї транзакції і закрив сесію. Закриту `AsyncSession` можна використати знову: вона тихо відкриває **нову** транзакцію, `INSERT` виконується, `RETURNING` дає 168 id… але `COMMIT` цієї транзакції ніхто не робить.

Найгірший вид помилки: **жодного винятку**, статус `done` — а даних немає.

Правило: фонова задача живе довше за запит, тож не бере в нього **нічого з часом життя запиту** — сесію бази, відкриті файли, об'єкт `Request`. Вона створює свої ресурси і сама робить `COMMIT`. Незмінні дані (id, параметри, JSON) передавати можна.

### Задача 3. Фоновий збір зі своєю сесією

Напиши `scrape_in_background_fixed(jobs, job_id)`: без репозиторію запиту — **своя** сесія (`async with SessionFactory() as session:`), `NewsRepository(session).add_many(valid)` і `await session.commit()`; потім `await jobs.update(job_id, status="done", news_saved=saved)`.

Ендпоінт `/api/scrape/jobs-fixed`, що запускає її у фоні, уже є в клітинці нижче. Перевірка — у наступній клітинці.

In [ ]:
async def scrape_in_background_fixed(jobs: JobStore, job_id: str) -> None:
    # YOUR CODE HERE
    pass


@app.post("/api/scrape/jobs-fixed", status_code=202)
async def start_fixed(background: BackgroundTasks, redis: RedisDep) -> dict[str, str]:
    job = await JobStore(redis).create("snapshot", "async")
    background.add_task(scrape_in_background_fixed, JobStore(redis), job.job_id)
    return {"job_id": job.job_id}

In [ ]:
reset_rate_limit()
client.delete("/api/news")
job_id = client.post("/api/scrape/jobs-fixed").json()["job_id"]
job = client.get(f"/api/scrape/jobs/{job_id}").json()
count = client.get("/api/news/count").json()["count"]
print("задача:", job["status"], job["news_saved"], "| у базі:", count)
assert job["status"] == "done", f"статус задачі — {job['status']}: функція має оновити його на done"
assert int(job["news_saved"]) == count == 168, "задача має зберегти всі 168 новин і зробити COMMIT"
print("✅ Задача 3 розв'язана")

---
## Самоперевірка

1. У якому порядку виконуються middleware для запиту і для відповіді?
2. Чому версію кешу збільшують після COMMIT, а не в ендпоінті? Чому `404` не кешують?
3. Чим небезпечні окремі `INCR` і `EXPIRE` у rate limit?
4. Що означає `202 Accepted` і як клієнт дізнається результат?
5. Чому фонова задача не може використати сесію бази з запиту?

<details>
<summary>Відповіді</summary>

1. Запит — від останнього зареєстрованого (зовнішнього) до першого; відповідь — навпаки.
2. COMMIT робить `get_db` після ендпоінта; версія до COMMIT дозволяє паралельному GET закешувати старі дані під новою версією. `404` у кеші приховав би новину, створену щойно, на весь TTL.
3. Збій між ними — ключ без TTL, лічильник більше не обнуляється, клієнта заблоковано назавжди. Рішення — одна транзакція і `EXPIRE … NX`.
4. «Прийнято до роботи», а не «готово»; у відповіді `job_id`, клієнт опитує `GET /api/scrape/jobs/{job_id}`.
5. Задача виконується після відповіді, коли сесію вже закомічено й закрито; записи в ній ніхто не закомітить — дані тихо пропадуть.

</details>

## Далі

- Книга курсу: [Урок 40](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_40/) — «цибуля» middleware покроково, чому інвалідація після COMMIT, межа фіксованого вікна, polling на справжньому сервері.
- У папці `news_hub`: `docker compose up -d db redis`, `alembic upgrade head`, `uvicorn news_hub.api:app --reload`; `pytest`.
- **Урок 42** — тести API агрегатора.

In [ ]:
client.__exit__(None, None, None)